# 01 Von der Frage zur strukturierten Antwort

LangChain durch Literate Programming lernen

## Konzept vor dem Code

    Für den Anfang reicht eine kleine Frage zu unserer LangChain-Aufgabe. Ich gebe den relevanten
    Text direkt mit. Eine automatische Dokumentensuche kommt erst in Notebook 03 hinzu.
    Ein Sprachmodell erhält bei jedem Aufruf Nachrichten. Die Systemnachricht legt die Aufgabe fest,
    die Human-Nachricht enthält die Frage, und die AI-Nachricht ist die Antwort. Ohne mitgesendeten
    Verlauf kennt der nächste Aufruf den vorherigen nicht.

    Ich stelle mir LCEL wie ein Fliessband vor: Jeder Schritt bekommt das Ergebnis des vorherigen.
    `Prompt → Nachrichten → ChatOpenAI → BFH API → Modell → AIMessage → Parser → Antwort`
    Der Operator `|` verbindet die Schritte. Ein Schema prüft die Form der Antwort. Es prüft noch
    nicht, ob ihr Inhalt stimmt. Im Demo-Modus steht an der Stelle des Modells ein selbst geschriebener
    Testwert. Das ist hilfreich für Codeprüfungen, ersetzt aber keinen Versuch mit einem LLM.

    Dokumentation: [ChatOpenAI](https://docs.langchain.com/oss/python/integrations/chat/openai),
    [BFH Inference](https://infra.pages.ti.bfh.ch/mlmp/src/llm/).

## Lernziele

Nach diesem Notebook kann ich:

- den Nachrichtenweg und die Rollen System Human und AI erklären
- einen variablen Prompt mit LCEL und einem Parser verbinden
- Streaming von einer vollständigen Antwort unterscheiden
- Tokenverbrauch einordnen und hypothetische Kosten berechnen

## Setup

Die folgende Zelle lädt die Einstellungen und zeigt die verwendeten Versionen. Das Modell wird erst in den Experimenten aufgerufen. So lässt sich ein Installationsproblem von einem Problem mit einer Modellantwort unterscheiden. Der Begriff hinter dem Experiment ist bereits oben eingeführt. Alle Notebooks laufen unabhängig in einem frischen Kernel.

In [1]:
import sys, os, json
from pathlib import Path
from importlib.metadata import version
from dotenv import load_dotenv
from modul_assistent.core import ROOT, mode, chat_model

load_dotenv(ROOT / '.env', override=True)
assert (3, 12) <= sys.version_info[:2] < (3, 13), 'Getestete Umgebung: Python 3.12'
for package in ['langchain-core', 'langchain-openai', 'langchain-classic']:
    print(package, version(package))
print('Betriebsart:', mode())

langchain-core 1.7.3
langchain-openai 1.7.3
langchain-classic 1.0.8
Betriebsart: demo


## Experiment

Zuerst definiere ich das Antwortschema. `antwort` und `quelle` sind getrennt, weil ich später die Herkunft gezielt prüfen möchte. Die Quelle ist hier ein manuell mitgegebener Auszug, kein Suchergebnis.

In [2]:
from modul_assistent.core import Antwort, answer_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
kontext = 'Gefordert sind vier eigenständig lauffähige Jupyter Notebooks. Quelle: LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1'
frage = 'Wie viele Notebooks sind gefordert?'
demo = Antwort(antwort='Gefordert sind vier Jupyter Notebooks.',
    quelle='LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1')
llm = chat_model([demo.model_dump_json()])
parser = PydanticOutputParser(pydantic_object=Antwort)

Die Chain ist absichtlich ausgeschrieben. Ich fülle zuerst die Variablen und sehe mir die Nachricht an. So erkenne ich einen fehlenden Kontext schon vor der Modellanfrage.

In [3]:
prompt = ChatPromptTemplate.from_messages([
    ('system', 'Beantworte die Frage auf Deutsch ausschliesslich aus dem Kontext. '
     'Fehlt die Information, sage es. {format}'),
    ('human', 'Kontext: {kontext}\nFrage: {frage}')
]).partial(format=parser.get_format_instructions())
daten = {'kontext': kontext, 'frage': frage}
nachrichten = prompt.invoke(daten)
print(nachrichten.to_string())
chain = prompt | llm | parser
ergebnis = chain.invoke(daten)
print(ergebnis.model_dump())
assert isinstance(ergebnis, Antwort)

System: Beantworte die Frage auf Deutsch ausschliesslich aus dem Kontext. Fehlt die Information, sage es. The output should be formatted as a JSON instance that conforms to the JSON schema below.

As an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}
the object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.

Here is the output schema:
```
{"properties": {"antwort": {"description": "Kurze deutsche Antwort oder ausdrückliches Nichtwissen", "title": "Antwort", "type": "string"}, "quelle": {"description": "Datei und Seite oder keine Quelle", "title": "Quelle", "type": "string"}}, "required": ["antwort", "quelle"]}
```
Human: Kontext: Gefordert sind vier eigenständig lauffähige Jupyter Notebooks. Quelle: LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1
Frage: Wie viele 

Beim Debuggen trenne ich die drei Schritte. Vor `llm.invoke` kann ich einen Haltepunkt setzen und die Nachricht prüfen. Anschliessend ist sichtbar, dass das Modell zunächst ein Nachrichtenobjekt liefert und der Parser daraus das Schema macht.

In [4]:
rohe_antwort = llm.invoke(nachrichten)
print(type(rohe_antwort).__name__, rohe_antwort.content)
print(parser.invoke(rohe_antwort))

AIMessage {"antwort":"Gefordert sind vier Jupyter Notebooks.","quelle":"LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1"}
antwort='Gefordert sind vier Jupyter Notebooks.' quelle='LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1'


Streaming liefert die Antwort stückweise. Ich streame hier die Modellnachricht und parse erst den zusammengesetzten Inhalt; ein halbes JSON-Dokument wäre noch nicht zuverlässig validierbar.

In [5]:
teile = []
for teil in llm.stream(nachrichten):
    if isinstance(teil.content, str):
        teile.append(teil.content)
        print(teil.content, end='')
print()
print(parser.parse(''.join(teile)))

{"antwort":"Gefordert sind vier Jupyter Notebooks.","quelle":"LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1"}
antwort='Gefordert sind vier Jupyter Notebooks.' quelle='LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1'


Für Tokenzahlen verwende ich nach Möglichkeit die vom Dienst gelieferten Nutzungsdaten. Ein lokal gewählter Tokenizer würde bei einem anderen Modell nur eine Näherung liefern. Die Beispielpreise unten sind frei gewählte Rechenwerte und ausdrücklich kein BFH-Tarif.

In [6]:
nutzung = rohe_antwort.usage_metadata
print('Vom Modell gemeldete Nutzung:', nutzung or 'im Demo-Modus nicht vorhanden')
def kosten(input_tokens, output_tokens, preis_input, preis_output):
    return (input_tokens * preis_input + output_tokens * preis_output) / 1_000_000
print('Hypothetische Kosten in CHF:', kosten(1000, 200, 0.20, 0.80))
assert abs(kosten(1000,200,0.20,0.80)-0.00036) < 1e-12

Vom Modell gemeldete Nutzung: im Demo-Modus nicht vorhanden
Hypothetische Kosten in CHF: 0.00036


## Häufige Fehler

Ich lasse die Variable `kontext` bewusst weg. Der Fehler entsteht vor dem Modellaufruf, weil die Vorlage nicht vollständig ausgefüllt werden kann. Ich fange genau den erwarteten Fehlertyp ab, damit das Notebook weiterläuft.

In [7]:
try:
    prompt.invoke({'frage': frage})
except KeyError as fehler:
    print(type(fehler).__name__, str(fehler)[:350])
else:
    raise AssertionError('Der fehlende Kontext wurde nicht erkannt')

KeyError "Input to ChatPromptTemplate is missing variables {'kontext'}.  Expected: ['frage', 'kontext'] Received: ['frage']\nNote: if you intended {kontext} to be part of the string and not a variable, please escape it with double curly braces like: '{{kontext}}'.\nFor troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/INVALID_PR


Die Korrektur ist, beide benötigten Werte zu übergeben. Zusätzlich zeige ich einen zweiten Fehler: Ein gültiger Text ist noch kein gültiges JSON. Der Parser macht diesen Unterschied sichtbar.

In [8]:
from langchain_core.exceptions import OutputParserException
assert prompt.invoke(daten)
try:
    parser.parse('Es sind vier Notebooks.')
except OutputParserException as fehler:
    print(type(fehler).__name__, str(fehler)[:180])
else:
    raise AssertionError('Der Parser akzeptiert unstrukturierten Text')
print(parser.parse(demo.model_dump_json()))

OutputParserException Invalid json output: Es sind vier Notebooks.
For troubleshooting, visit: https://docs.langchain.com/oss/python/langchain/errors/OUTPUT_PARSING_FAILURE 
antwort='Gefordert sind vier Jupyter Notebooks.' quelle='LangChain_Literate_Programming_Aufgabenstellung.pdf, S. 1'


## Prompt Logbuch

**Mein eigener erster Prompt ohne KI-Hilfe:**

> Noch von mir einzutragen. Die folgende Endversion ist ein KI-gestützter Vorschlag. Ein nachträglich erfundener Erstprompt würde die geforderte eigene Iteration nicht belegen.

**Vorgeschlagene Endversion:**

```text
Beantworte die Frage nur aus dem Kontext. Gib JSON mit antwort und quelle zurück. Fehlt die Information, sage dies ausdrücklich und verwende keine erfundene Quelle.
```

**Begründung des Vorschlags:** Der Kontext begrenzt die Informationsbasis. Das Schema macht die Antwort für weitere Verarbeitung nutzbar. Ob ein engerer Prompt tatsächlich bessere Ergebnisse liefert, muss ich mit beiden Versionen am BFH-Modell prüfen.

**Mein Vergleich nach dem eigenen Versuch:** Hier trage ich Modell, Datum, beide tatsächlichen Antworten und meine Beobachtung ein. Demo-Ausgaben sind kein Modellversuch.

## Zusammenfassung

Die erste Frage wird strukturiert beantwortet. Nachrichtenrollen, Prompt, Modell und Parser sind getrennte Schritte. Ein Schemafehler ist kein Wissensfehler. In Notebook 02 bleibt die Frage erhalten, ob der Assistent sich an eine zweite Nachricht erinnern kann.

**Selbstkontrolle:** Kann ich die oben genannten Lernziele anhand der ausgeführten Zellen erklären, ohne die Hilfsfunktionen nur abzulesen?

## Übungsaufgabe

Ergänze eine Frage, deren Antwort im Kontext fehlt. Prüfe im BFH-Modus, ob das Modell Nichtwissen ausdrückt. Im Demo-Modus schreibe dafür einen zusätzlichen klar gekennzeichneten Testwert. Begründe, was dieser Test belegen kann und was nicht.

Die folgende Zelle bleibt absichtlich leer und ist für meine eigene Lösung vorgesehen.